In [6]:
import numpy as np
import pandas as pd


# =========================
# 1. 读取 SWC
# =========================
def load_swc(path):
    swc_df = pd.read_csv(
        path,
        comment='#',
        sep=r'\s+',
        names=['id','type','x','y','z','r','parent']
    )
    swc_df = swc_df.loc[swc_df.type.isin([1,3])]
    return swc_df


# =========================
# 2. 自动检测 soma
# =========================
def get_soma(swc_df):
    soma_nodes = swc_df[swc_df['type'] == 1]

    if len(soma_nodes) > 0:
        node = soma_nodes.iloc[0]
    else:
        # fallback: root node
        node = swc_df[swc_df['parent'] == -1].iloc[0]

    return np.array([node['x'], node['y'], node['z']])


# =========================
# 3. SWC → edges
# =========================
def swc_to_edges(swc_df):
    nodes = swc_df.set_index('id').to_dict('index')

    edges = []
    for _, row in swc_df.iterrows():
        if row['parent'] == -1:
            continue

        p1 = np.array([row['x'], row['y'], row['z']])
        parent = nodes[row['parent']]
        p2 = np.array([parent['x'], parent['y'], parent['z']])

        edges.append((p1, p2))

    return edges


# =========================
# 4. 线段-球面求交（3D）
# =========================
def line_sphere_intersections(p1, p2, c, r):
    p1, p2, c = map(np.array, (p1, p2, c))
    d = p2 - p1

    a = np.dot(d, d)
    b = 2 * np.dot(d, p1 - c)
    c_coef = np.dot(p1 - c, p1 - c) - r**2

    disc = b**2 - 4*a*c_coef
    if disc < 0:
        return []

    sqrt_disc = np.sqrt(disc)

    t1 = (-b - sqrt_disc) / (2*a)
    t2 = (-b + sqrt_disc) / (2*a)

    pts = []
    for t in [t1, t2]:
        if 0 <= t <= 1:
            pts.append(p1 + t * d)

    return pts


# =========================
# 5. R = 100 Sholl intersections
# =========================
def sholl_radius_100_points(swc_df, soma,R=100):
    # R = 100
    edges = swc_to_edges(swc_df)

    pts_all = []

    for p1, p2 in edges:
        pts = line_sphere_intersections(p1, p2, soma, R)
        pts_all.extend(pts)

    return np.array(pts_all)


# =========================
# 6. 一体化运行
# =========================
def run_sholl_pipeline(swc_path,R=100):
    # 1) load
    swc_df = load_swc(swc_path)
    R = R
    # 2) soma
    soma = get_soma(swc_df)
    print("soma:", soma)

    # 3) intersection
    pts = sholl_radius_100_points(swc_df, soma,R=R)

    # print("intersection points shape:", pts.shape)
    # print("first 5 points:\n", pts[:5])

    return soma, pts


import json
import numpy as np

def save_sholl_points_to_json(filepath, pts, soma, radius=100):
    data = {
        "radius": radius,
        "soma": soma.tolist() if isinstance(soma, np.ndarray) else list(soma),
        "points": pts.tolist() if isinstance(pts, np.ndarray) else pts
    }

    with open(filepath, "w") as f:
        json.dump(data, f, indent=2)

    print(f"Saved to {filepath}")

# =========================
# Example usage
# =========================
# soma, pts = run_sholl_pipeline("your_neuron.swc")




In [ ]:
import os,glob
os.chdir(r"J:\BLA_three_types\soma_sholl")

# df = load_swc("251038_001.swc")
swcs = glob.glob(r"J:\BLA_three_types\soma_sholl\BLA_swc_mirrow\*")
for file in swcs:
    radius =50
    try:
        filepath = file.split("\\")[-1][0:-4]+".json"
        
        soma, pts = run_sholl_pipeline(file,R=radius)
        save_sholl_points_to_json(filepath, pts, soma, radius=radius)
    except:
        print("file")

soma: [7205.   5497.94 2557.34]
Saved to 221058_038_Sst.json
soma: [7266.7  5684.62 2682.34]
Saved to 221058_040_Sst.json
soma: [7371.48 5721.3  2808.22]
Saved to 221058_042_Sst.json
soma: [7135.78 6000.8  2624.22]
Saved to 221058_071_Sst.json
soma: [6855.58 5953.14 2781.3 ]
Saved to 221058_104_Sst.json
soma: [6826.4  5614.94 2611.18]
Saved to 221058_107_Sst.json
soma: [7268.26 5856.38 2561.14]
Saved to 221058_113_Sst.json
soma: [6754.98 5947.02 2779.82]
Saved to 221297_036_Sst.json
soma: [6802.7  5947.16 2817.84]
Saved to 221297_037_Sst.json
soma: [6734.06 5933.4  2757.06]
Saved to 221297_038_Sst.json
soma: [6917.82 5710.54 2685.98]
Saved to 230058_073_Sst.json
soma: [6996.   5531.66 2570.68]
Saved to 230058_074_Sst.json
soma: [7097.92 5525.34 2562.34]
Saved to 230058_076_Sst.json
soma: [6900.98 6027.1  2766.46]
Saved to 230058_078_Sst.json
soma: [6427.8  6134.66 2725.54]
Saved to 234172_015_Crh.json
soma: [6733.16 5836.2  2758.98]
Saved to 234172_016_Crh.json
soma: [6571.28 6070.98 2

: 

In [ ]:
### get dendrite edge

In [3]:
files = glob.glob("J:\BLA_three_types\csv_terminal\*")

In [ ]:
file

In [4]:
outpath = r"J:\BLA_three_types\soma_sholl\dendrite_edge"
for file in files:
    dftmp = pd.read_csv(file,index_col=0)
    dftmp = dftmp.loc[dftmp.type.isin([1,3])]
    outfile = os.path.join(outpath,os.path.basename(file))
    dftmp.to_csv(outfile)



In [5]:
import numpy as np
import plotly.graph_objects as go

def plot_sphere(soma, radius=100, resolution=50):
    x0, y0, z0 = soma

    # 球面参数
    theta = np.linspace(0, np.pi, resolution)
    phi = np.linspace(0, 2*np.pi, resolution)

    theta, phi = np.meshgrid(theta, phi)

    x = x0 + radius * np.sin(theta) * np.cos(phi)
    y = y0 + radius * np.sin(theta) * np.sin(phi)
    z = z0 + radius * np.cos(theta)

    fig = go.Figure()

    fig.add_trace(go.Surface(
        x=x,
        y=y,
        z=z,
        surfacecolor=np.zeros_like(x),
        colorscale=[[0, 'skyblue'], [1, 'skyblue']],
        opacity=0.3,
        showscale=False,
        name="Sholl Sphere (R=100)"
    ))

    fig.update_layout(
        scene=dict(
            xaxis_title='X',
            yaxis_title='Y',
            zaxis_title='Z',
            aspectmode='data'
        ),
        title="3D Sholl Sphere (R=100)"
    )

    fig.show(renderer="browser")

In [6]:
soma = np.array([100, 200, 150])

plot_sphere(soma, radius=100)

In [ ]:
os.path.basename(file)

In [ ]:
filenew = os.basename(file)

In [ ]:
dftmp.loc[dftmp.type ==3]

In [ ]:
import numpy as np

def line_sphere_intersections(p1, p2, c, r):
    p1 = np.array(p1)
    p2 = np.array(p2)
    c  = np.array(c)

    d = p2 - p1

    a = np.dot(d, d)
    b = 2 * np.dot(d, p1 - c)
    c_coef = np.dot(p1 - c, p1 - c) - r**2

    disc = b**2 - 4*a*c_coef

    if disc < 0:
        return []

    sqrt_disc = np.sqrt(disc)

    t1 = (-b - sqrt_disc) / (2*a)
    t2 = (-b + sqrt_disc) / (2*a)

    pts = []

    for t in [t1, t2]:
        if 0 <= t <= 1:
            pt = p1 + t * d
            pts.append(tuple(pt))

    return pts

In [ ]:
def swc_to_edges(swc_df):
    nodes = swc_df.set_index('id').to_dict('index')

    edges = []
    for _, row in swc_df.iterrows():
        if row['parent'] == -1:
            continue

        p1 = (row['x'], row['y'], row['z'])
        p2 = (
            nodes[row['parent']]['x'],
            nodes[row['parent']]['y'],
            nodes[row['parent']]['z']
        )

        edges.append((p1, p2))

    return edges

In [ ]:
def sholl_points_single_radius(swc_df, soma, radius):
    edges = swc_to_edges(swc_df)
    soma = np.array(soma)

    pts = []

    for p1, p2 in edges:
        pts.extend(line_sphere_intersections(p1, p2, soma, radius))

    return np.array(pts)

In [ ]:
intersection_points

In [ ]:
plot_showcase(df, intersection_points, radius=100)